<a href="https://colab.research.google.com/github/MPMauricio/Calendarizacion-Ujieres-Antigravity2026/blob/main/MapaFinalCM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# ============================================
# ✅ CORRECCIÓN: "Numero de Recarga" visible
# ============================================
!pip install pandas openpyxl -q

import pandas as pd
from datetime import datetime
import math
import json
import os
import zipfile

# ────────────────────────────────────────────
# FUNCIONES AUXILIARES
# ────────────────────────────────────────────
def calcular_distancia_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1_rad, lon1_rad = math.radians(lat1), math.radians(lon1)
    lat2_rad, lon2_rad = math.radians(lat2), math.radians(lon2)
    dlon, dlat = lon2_rad - lon1_rad, lat2_rad - lat1_rad
    a = math.sin(dlat/2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon/2)**2
    return round(R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a)), 2)

def formatear_dolares(valor):
    try:
        if pd.isna(valor) or str(valor) in ['ERROR:#N/A', 'N/A', '', '0']:
            return "$0.00"
        limpio = str(valor).replace('$', '').replace(',', '').strip()
        return f"${float(limpio):,.2f}"
    except:
        return "$0.00"

def limpiar_texto(valor):
    if pd.isna(valor) or str(valor) in ['ERROR:#N/A', 'N/A', '', 'nan']:
        return 'N/A'
    return str(valor).strip()

# ────────────────────────────────────────────
# COORDENADAS DE SEDES
# ────────────────────────────────────────────
COORDENADAS_SEDES = {
    'SANTAANA2': {'lat': 13.988190, 'lon': -89.548592, 'nombre': 'SANTAANA2'},
    'AHUACHAP': {'lat': 13.920802, 'lon': -89.845185, 'nombre': 'AHUACHAP'},
    'SONSONATE': {'lat': 13.716444, 'lon': -89.722560, 'nombre': 'SONSONATE'},
    'SSNORTE': {'lat': 13.700532, 'lon': -89.151834, 'nombre': 'SSNORTE'},
    'SSSUR': {'lat': 13.685925, 'lon': -89.226888, 'nombre': 'SSSUR'}
}

# ────────────────────────────────────────────
# 1. CARGA DE DATOS
# ────────────────────────────────────────────
print("="*70)
print("📂 CARGA DE ARCHIVO EXCEL - CORRECCIÓN NUMERO DE RECARGA")
print("="*70)

from google.colab import files
print("\n📤 Selecciona tu archivo Excel:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

df_temp = pd.read_excel(filename)
df_temp = df_temp.dropna(how='all')

# Mapeo corregido para "Numero de  Recarga"
mapeo_cols = {
    'Ruta': 'Ruta',
    'Departamento': 'Departamento',
    'Frecuencia de visita': 'Frecuencia de visita',
    'Dias de visita': 'Dias de visita',
    'Categoria': 'Categoria',
    'Codigo de PDV': 'Codigo de PDV',
    'Numero de Recarga': 'Numero de  Recarga', # ✅ AQUÍ ESTABA EL ERROR
    'Distancia': 'Distancia',
    'Nombre del PDV': 'Nombre del PDV',
    'Venta Promedio': 'Venta Promedio',
    'Latitud': 'Latitud',
    'Longitud': 'Longitud'
}

cols_existentes = df_temp.columns.tolist()
df_limpio = pd.DataFrame()

for nuevo, original in mapeo_cols.items():
    col_encontrada = None
    for col in cols_existentes:
        # Compara ignorando mayúsculas y espacios extra
        if col.strip().lower().replace('  ', ' ') == original.strip().lower().replace('  ', ' '):
            col_encontrada = col
            break
    if col_encontrada:
        df_limpio[nuevo] = df_temp[col_encontrada]
    else:
        # Si no encuentra la columna exacta, busca una que sea similar
        for col in cols_existentes:
            if 'Numero' in col and 'Recarga' in col:
                col_encontrada = col
                df_limpio[nuevo] = df_temp[col_encontrada]
                break
        if not col_encontrada:
            df_limpio[nuevo] = 'N/A'

df_limpio['Latitud'] = pd.to_numeric(df_limpio['Latitud'], errors='coerce')
df_limpio['Longitud'] = pd.to_numeric(df_limpio['Longitud'], errors='coerce')
df_limpio = df_limpio.dropna(subset=['Latitud', 'Longitud'])

df_limpio['Venta Promedio Formato'] = df_limpio['Venta Promedio'].apply(formatear_dolares)
df_limpio['Venta Promedio Num'] = pd.to_numeric(
    df_limpio['Venta Promedio'].astype(str).str.replace(r'[^\d.]', '', regex=True),
    errors='coerce'
).fillna(0)

def determinar_sede(row):
    lat = row['Latitud']
    lon = row['Longitud']
    sede_cercana = None
    min_dist = float('inf')
    for sede_nombre, sede_coords in COORDENADAS_SEDES.items():
        dist = calcular_distancia_km(lat, lon, sede_coords['lat'], sede_coords['lon'])
        if dist < min_dist:
            min_dist = dist
            sede_cercana = sede_nombre
    return sede_cercana

df_limpio['Sede'] = df_limpio.apply(determinar_sede, axis=1)

total_pdv = len(df_limpio)
print(f"\n✅ {total_pdv} PDV cargados correctamente")

departamentos_unicos = sorted(df_limpio['Departamento'].dropna().unique().tolist())
rutas_unicas = sorted(df_limpio['Ruta'].dropna().unique().tolist())
frecuencias_unicas = sorted(df_limpio['Frecuencia de visita'].dropna().unique().tolist())

depto_rutas = {}
for depto in departamentos_unicos:
    rutas_depto = sorted(df_limpio[df_limpio['Departamento'] == depto]['Ruta'].dropna().unique().tolist())
    depto_rutas[depto] = rutas_depto

# ────────────────────────────────────────────
# 2. PREPARAR DATOS
# ───────────────────────────────────────────
colores_rutas = {}
colores_disponibles = [
    '#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7',
    '#DDA0DD', '#98D8C8', '#F7DC6F', '#BB8FCE', '#85C1E2',
    '#F1948A', '#7DCEA0', '#F4D03F', '#A9DFBF', '#F5B7B1',
    '#AED6F1', '#D7BDE2', '#A3E4D7', '#F9E79F', '#FADBD8'
]

for i, ruta in enumerate(rutas_unicas):
    colores_rutas[ruta] = colores_disponibles[i % len(colores_disponibles)]

pdv_data = []
for _, row in df_limpio.iterrows():
    sede_asignada = row['Sede']
    sede_coords = COORDENADAS_SEDES.get(sede_asignada, COORDENADAS_SEDES['SANTAANA2'])

    pdv_data.append({
        'ruta': limpiar_texto(row['Ruta']),
        'departamento': limpiar_texto(row['Departamento']),
        'frecuencia': limpiar_texto(row['Frecuencia de visita']),
        'codigo_pdv': limpiar_texto(row['Codigo de PDV']),
        'numero_recarga': limpiar_texto(row['Numero de Recarga']), # ✅ Campo corregido
        'venta_promedio': row['Venta Promedio Formato'],
        'venta_promedio_num': float(row['Venta Promedio Num']),
        'lat': float(row['Latitud']),
        'lon': float(row['Longitud']),
        'sede': sede_asignada,
        'sede_nombre': sede_coords['nombre'],
        'color_ruta': colores_rutas.get(row['Ruta'], '#667eea')
    })

totales_por_ruta = {}
for ruta in rutas_unicas:
    total = sum(p['venta_promedio_num'] for p in pdv_data if p['ruta'] == ruta)
    totales_por_ruta[ruta] = f"${total:,.2f}"

# ────────────────────────────────────────────
# 3. CREAR HTML CON "NUMERO DE RECARGA" VISIBLE
# ───────────────────────────────────────────
html_content = f"""
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Mapa PDV - Ruteo Final</title>
    <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
    <style>
        * {{ margin: 0; padding: 0; box-sizing: border-box; }}
        body {{ font-family: 'Segoe UI', sans-serif; overflow: hidden; }}
        #map {{ height: 100vh; width: 100%; z-index: 1; }}

        .panel {{
            position: fixed; z-index: 1000; background: white; border-radius: 12px;
            box-shadow: 0 4px 20px rgba(0,0,0,0.2); overflow: hidden;
        }}
        .filters {{ top: 10px; left: 10px; max-width: 300px; width: calc(100% - 20px); max-height: 90vh; display: flex; flex-direction: column; }}
        .stats {{ top: 10px; right: 10px; max-width: 300px; width: calc(100% - 20px); }}
        .route-info {{ bottom: 10px; left: 10px; right: 10px; max-height: 35vh; display: none; }}
        .minimized {{ max-height: 50px !important; }}
        .hidden {{ display: none !important; }}

        .panel-head {{
            padding: 14px; color: white; cursor: pointer; display: flex;
            justify-content: space-between; align-items: center; font-weight: 600;
        }}
        .filters .panel-head {{ background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); }}
        .stats .panel-head {{ background: linear-gradient(135deg, #fa709a 0%, #fee140 100%); }}
        .route-info .panel-head {{ background: linear-gradient(135deg, #4facfe 0%, #00f2fe 100%); }}

        .panel-body {{ padding: 15px; overflow-y: auto; flex: 1; }}
        .minimized .panel-body {{ display: none; }}

        .acc-item {{ margin-bottom: 10px; border: 1px solid #eee; border-radius: 8px; overflow: hidden; }}
        .acc-head {{
            padding: 10px; background: #f8f9fa; cursor: pointer; font-weight: 600;
            display: flex; justify-content: space-between; align-items: center;
        }}
        .acc-head:hover {{ background: #e9ecef; }}
        .acc-body {{ max-height: 200px; overflow-y: auto; transition: max-height 0.3s; background: white; }}
        .acc-body.collapsed {{ max-height: 0; }}

        .chk-row {{ padding: 8px; display: flex; align-items: center; border-bottom: 1px solid #f0f0f0; }}
        .chk-row:hover {{ background: #f8f9fa; }}
        .chk-row input {{ margin-right: 10px; transform: scale(1.2); accent-color: #667eea; }}

        .btn {{ width: 100%; padding: 12px; border: none; border-radius: 8px; font-weight: 600; cursor: pointer; margin-top: 8px; color: white; }}
        .btn-apply {{ background: linear-gradient(135deg, #11998e 0%, #38ef7d 100%); }}
        .btn-route {{ background: linear-gradient(135deg, #00b09b 0%, #96c93d 100%); display: none; }}
        .btn-route.show {{ display: block; }}
        .btn-reset {{ background: #95a5a6; }}

        .stat-box {{ background: #f8f9fa; padding: 10px; border-radius: 8px; margin-bottom: 10px; border-left: 4px solid #667eea; }}
        .stat-val {{ font-size: 20px; font-weight: 700; color: #2c3e50; }}
        .stat-val.money {{ color: #27ae60; }}

        .route-step {{ display: flex; align-items: center; padding: 8px; border-bottom: 1px solid #eee; font-size: 13px; }}
        .step-num {{ width: 24px; height: 24px; background: #667eea; color: white; border-radius: 50%; display: flex; align-items: center; justify-content: center; margin-right: 10px; font-size: 12px; font-weight: bold; }}
        .route-start .step-num {{ background: #27ae60; }}
        .route-end .step-num {{ background: #e74c3c; }}

        /* Popup personalizado */
        .custom-popup .leaflet-popup-content-wrapper {{
            border-radius: 10px; padding: 0; overflow: hidden; box-shadow: 0 5px 15px rgba(0,0,0,0.2);
        }}
        .custom-popup .leaflet-popup-content {{ margin: 0; width: 260px !important; }}
        .popup-header {{
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white; padding: 10px 12px; font-weight: 600; font-size: 14px;
        }}
        .popup-body {{ padding: 12px; background: white; font-size: 13px; line-height: 1.6; }}
        .popup-row {{ display: flex; justify-content: space-between; margin-bottom: 6px; border-bottom: 1px dashed #eee; padding-bottom: 4px; }}
        .popup-row:last-child {{ border-bottom: none; margin-bottom: 0; }}
        .popup-label {{ color: #7f8c8d; font-weight: 500; }}
        .popup-val {{ color: #2c3e50; font-weight: 600; text-align: right; max-width: 60%; word-break: break-word; }}
        .popup-val.money {{ color: #27ae60; }}
        .popup-val.route {{ color: #667eea; }}
        .popup-order {{
            margin-top: 10px; background: #f8f9fa; padding: 6px; border-radius: 6px;
            text-align: center; font-weight: bold; color: #e74c3c; border: 1px dashed #e74c3c;
        }}
    </style>
</head>
<body>
    <div id="map"></div>

    <div class="panel filters" id="filtersPanel">
        <div class="panel-head" onclick="togglePanel('filters')"> Filtros <span>🔼</span></div>
        <div class="panel-body">
            <div class="acc-item">
                <div class="acc-head" onclick="toggleAcc('deptoAcc')">📍 Departamento <span>▼</span></div>
                <div class="acc-body" id="deptoAcc">
                    {''.join([f'<div class="chk-row"><input type="checkbox" class="chk-depto" id="d_{i}" value="{d}" onchange="updateRutas()"><label for="d_{i}">{d}</label></div>' for i, d in enumerate(departamentos_unicos)])}
                </div>
            </div>
            <div class="acc-item">
                <div class="acc-head" onclick="toggleAcc('rutaAcc')">🛣️ Ruta <span>▼</span></div>
                <div class="acc-body" id="rutaAcc">
                    <div id="listaRutas" style="padding:10px; color:#999; text-align:center;">Selecciona depto</div>
                </div>
            </div>
            <div class="acc-item">
                <div class="acc-head" onclick="toggleAcc('freqAcc')">🔄 Frecuencia <span>▼</span></div>
                <div class="acc-body" id="freqAcc">
                    {''.join([f'<div class="chk-row"><input type="checkbox" class="chk-freq" id="f_{i}" value="{f}"><label for="f_{i}">{f}</label></div>' for i, f in enumerate(frecuencias_unicas)])}
                </div>
            </div>

            <button class="btn btn-apply" onclick="aplicar()">✅ Aplicar</button>
            <button class="btn btn-route" id="btnRuta" onclick="trazarRuta()">🗺️ Ver Ruta Optimizada</button>
            <button class="btn btn-reset" onclick="resetear()">🔄 Limpiar</button>
        </div>
    </div>

    <div class="panel stats" id="statsPanel">
        <div class="panel-head" onclick="togglePanel('stats')"> Estadísticas <span>🔼</span></div>
        <div class="panel-body">
            <div class="stat-box"><div style="font-size:12px; color:#777;">PDV Visibles</div><div class="stat-val" id="stVis">{total_pdv}</div></div>
            <div class="stat-box"><div style="font-size:12px; color:#777;">Total Venta</div><div class="stat-val money" id="stTot">${sum(p['venta_promedio_num'] for p in pdv_data):,.2f}</div></div>
            <div id="stRutas" style="margin-top:10px; max-height:150px; overflow-y:auto;"></div>
        </div>
    </div>

    <div class="panel route-info" id="routePanel">
        <div class="panel-head" onclick="togglePanel('route')">️ Ruta del Día <span>🔼</span></div>
        <div class="panel-body">
            <div style="display:flex; gap:10px; margin-bottom:10px;">
                <div class="stat-box" style="flex:1; text-align:center;"><div class="stat-val" id="rDist">-</div><div style="font-size:10px;">Km Total</div></div>
                <div class="stat-box" style="flex:1; text-align:center;"><div class="stat-val" id="rTime">-</div><div style="font-size:10px;">Tiempo Est.</div></div>
            </div>
            <div id="rList" style="max-height: 150px; overflow-y: auto;"></div>
        </div>
    </div>

    <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
    <script>
        const dataOrig = {json.dumps(pdv_data)};
        const rutasMap = {json.dumps(depto_rutas)};
        const colores = {json.dumps(colores_rutas)};
        const sedes = {json.dumps(COORDENADAS_SEDES)};
        const totales = {json.dumps(totales_por_ruta)};

        let data = [...dataOrig];
        let map, capaMarcadores, capaRuta;

        function init() {{
            map = L.map('map').setView([13.8, -89.2], 9);
            L.tileLayer('https://{{s}}.basemaps.cartocdn.com/light_all/{{z}}/{{x}}/{{y}}.png', {{maxZoom: 19}}).addTo(map);
            pintar(data);
            stats(data);
        }}

        function togglePanel(id) {{ document.getElementById(id+'Panel').classList.toggle('minimized'); }}
        function toggleAcc(id) {{ document.getElementById(id).classList.toggle('collapsed'); }}

        function updateRutas() {{
            const depts = Array.from(document.querySelectorAll('.chk-depto:checked')).map(c => c.value);
            const box = document.getElementById('listaRutas');
            let rutas = new Set();
            if(depts.length === 0) Object.keys(colores).forEach(r => rutas.add(r));
            else depts.forEach(d => (rutasMap[d]||[]).forEach(r => rutas.add(r)));

            box.innerHTML = Array.from(rutas).sort().map((r,i) =>
                `<div class="chk-row"><input type="checkbox" class="chk-ruta" value="${{r}}" id="r_${{i}}"><label for="r_${{i}}">${{r}}</label></div>`
            ).join('');
        }}

        function pintar(datos) {{
            if(capaMarcadores) map.removeLayer(capaMarcadores);
            if(capaRuta) map.removeLayer(capaRuta);

            capaMarcadores = L.layerGroup();
            datos.forEach(p => {{
                const marker = L.circleMarker([p.lat, p.lon], {{
                    radius: 8, fillColor: p.color_ruta, color: "#fff", weight: 2, fillOpacity: 0.9
                }});
                marker.bindPopup(`
                    <div class="popup-header">${{p.nombre_pdv}}</div>
                    <div class="popup-body">
                        <div class="popup-row"><span class="popup-label">Código:</span><span class="popup-val">${{p.codigo_pdv}}</span></div>
                        <div class="popup-row"><span class="popup-label">N° Recarga:</span><span class="popup-val">${{p.numero_recarga}}</span></div>
                        <div class="popup-row"><span class="popup-label">Venta:</span><span class="popup-val money">${{p.venta_promedio}}</span></div>
                    </div>
                `, {{className: 'custom-popup'}});
                capaMarcadores.addLayer(marker);
            }});
            capaMarcadores.addTo(map);
        }}

        function trazarRuta() {{
            if(data.length === 0) return alert("Selecciona datos primero");

            const countSedes = {{}};
            data.forEach(p => countSedes[p.sede] = (countSedes[p.sede]||0)+1);
            const sedeKey = Object.keys(countSedes).sort((a,b)=>countSedes[b]-countSedes[a])[0];
            const sede = sedes[sedeKey];

            let pendientes = [...data];
            let ruta = [];
            let lat = sede.lat, lon = sede.lon, distTotal = 0;

            while(pendientes.length > 0) {{
                let mejorIdx = -1, mejorDist = Infinity;
                pendientes.forEach((p, idx) => {{
                    const d = Math.sqrt(Math.pow(p.lat-lat,2) + Math.pow(p.lon-lon,2));
                    if(d < mejorDist) {{ mejorDist = d; mejorIdx = idx; }}
                }});
                const p = pendientes.splice(mejorIdx, 1)[0];
                const distKm = 6371 * 2 * Math.asin(Math.sqrt(Math.pow(Math.sin((p.lat-lat)*Math.PI/360),2) + Math.cos(lat*Math.PI/180)*Math.cos(p.lat*Math.PI/180)*Math.pow(Math.sin((p.lon-lon)*Math.PI/360),2)));
                ruta.push({{...p, distPrev: distKm}});
                distTotal += distKm;
                lat = p.lat; lon = p.lon;
            }}
            const distRet = 6371 * 2 * Math.asin(Math.sqrt(Math.pow(Math.sin((sede.lat-lat)*Math.PI/360),2) + Math.cos(lat*Math.PI/180)*Math.cos(sede.lat*Math.PI/180)*Math.pow(Math.sin((sede.lon-lon)*Math.PI/360),2)));
            distTotal += distRet;

            // Limpiar capa anterior
            if(capaRuta) map.removeLayer(capaRuta);
            capaRuta = L.layerGroup();

            // Dibujar línea roja
            const coords = [[sede.lat, sede.lon], ...ruta.map(p=>[p.lat, p.lon]), [sede.lat, sede.lon]];
            const polyline = L.polyline(coords, {{color: '#e74c3c', weight: 5, opacity: 0.85, lineCap: 'round'}});
            capaRuta.addLayer(polyline);

            // Marcador Sede
            const mSede = L.circleMarker([sede.lat, sede.lon], {{radius: 10, fillColor: '#27ae60', color: 'white', weight: 3}}).bindPopup(` Sede: ${{sede.nombre}}`, {{className: 'custom-popup'}});
            capaRuta.addLayer(mSede);

            // Marcadores numerados CON POPUP COMPLETO Y NUMERO DE RECARGA
            ruta.forEach((p, i) => {{
                const popupHtml = `
                    <div class="popup-header">${{p.nombre_pdv}}</div>
                    <div class="popup-body">
                        <div class="popup-row"><span class="popup-label">Código PDV:</span><span class="popup-val">${{p.codigo_pdv}}</span></div>
                        <div class="popup-row"><span class="popup-label">N° Recarga:</span><span class="popup-val" style="color:#2980b9; font-weight:bold;">${{p.numero_recarga}}</span></div>
                        <div class="popup-row"><span class="popup-label">Ruta:</span><span class="popup-val route">${{p.ruta}}</span></div>
                        <div class="popup-row"><span class="popup-label">Venta Promedio:</span><span class="popup-val money">${{p.venta_promedio}}</span></div>
                        <div class="popup-order">📍 Orden de visita: #${{i + 1}}</div>
                    </div>
                `;

                const numMarker = L.marker([p.lat, p.lon], {{
                    icon: L.divIcon({{
                        className: '',
                        html: `<div style="background:white; color:#e74c3c; border:3px solid #e74c3c; border-radius:50%; width:28px; height:28px; display:flex; align-items:center; justify-content:center; font-weight:800; font-size:14px; box-shadow: 0 3px 8px rgba(0,0,0,0.4); cursor: pointer; transition: transform 0.2s;">${{i+1}}</div>`,
                        iconSize: [28, 28],
                        iconAnchor: [14, 14]
                    }})
                }}).bindPopup(popupHtml, {{className: 'custom-popup', maxWidth: 280}});

                capaRuta.addLayer(numMarker);
            }});

            capaRuta.addTo(map);
            map.fitBounds(polyline.getBounds().pad(0.15));

            // Actualizar UI
            document.getElementById('routePanel').classList.remove('hidden');
            document.getElementById('rDist').innerText = distTotal.toFixed(1) + " km";
            document.getElementById('rTime').innerText = Math.ceil((distTotal/30)*60 + ruta.length*10) + " min";

            document.getElementById('rList').innerHTML = `
                <div class="route-step route-start"><div class="step-num">🏢</div><div>Inicio: ${{sede.nombre}} (8:00 AM)</div></div>
                ${{ruta.map((p,i) => `<div class="route-step"><div class="step-num">${{i+1}}</div><div>${{p.nombre_pdv}} <small style="color:#7f8c8d">(${{p.distPrev.toFixed(2)}} km)</small></div></div>`).join('')}}
                <div class="route-step route-end"><div class="step-num">🏁</div><div>Retorno: ${{sede.nombre}}</div></div>
            `;
        }}

        function aplicar() {{
            const d = Array.from(document.querySelectorAll('.chk-depto:checked')).map(c=>c.value);
            const r = Array.from(document.querySelectorAll('.chk-ruta:checked')).map(c=>c.value);
            const f = Array.from(document.querySelectorAll('.chk-freq:checked')).map(c=>c.value);

            data = dataOrig.filter(p =>
                (d.length===0 || d.includes(p.departamento)) &&
                (r.length===0 || r.includes(p.ruta)) &&
                (f.length===0 || f.includes(p.frecuencia))
            );

            pintar(data);
            stats(data);
            document.getElementById('btnRuta').classList.toggle('show', data.length > 0);
            document.getElementById('routePanel').classList.add('hidden');
            if(capaRuta) map.removeLayer(capaRuta);

            if(data.length > 0) map.fitBounds(L.latLngBounds(data.map(p=>[p.lat, p.lon])).pad(0.1));
        }}

        function stats(datos) {{
            document.getElementById('stVis').innerText = datos.length;
            const tot = datos.reduce((s,p)=>s+p.venta_promedio_num, 0);
            document.getElementById('stTot').innerText = "$"+tot.toLocaleString('en-US', {{minimumFractionDigits:2}});

            const rConDatos = [...new Set(datos.map(p=>p.ruta))];
            document.getElementById('stRutas').innerHTML = rConDatos.map(r =>
                `<div style="display:flex;justify-content:space-between;padding:5px 0;border-bottom:1px solid #eee;">
                    <span><span style="display:inline-block;width:10px;height:10px;background:${{colores[r]}};border-radius:50%;margin-right:5px;"></span>${{r}}</span>
                    <b>${{totales[r]}}</b>
                </div>`
            ).join('');
        }}

        function resetear() {{
            document.querySelectorAll('input').forEach(c=>c.checked=false);
            data = [...dataOrig];
            updateRutas();
            aplicar();
        }}

        window.onload = init;
    </script>
</body>
</html>
"""

# ────────────────────────────────────────────
# 4. GUARDAR Y EXPORTAR
# ────────────────────────────────────────────
carpeta_netlify = "sitio_mapa_pdv_recarga_ok"
os.makedirs(carpeta_netlify, exist_ok=True)

archivo_index = os.path.join(carpeta_netlify, "index.html")
with open(archivo_index, 'w', encoding='utf-8') as f:
    f.write(html_content)

print(f"\n✅ ARCHIVO CREADO CON NUMERO DE RECARGA: {carpeta_netlify}/index.html")

nombre_zip = f"Mapa_PDV_Recarga_OK_{datetime.now().strftime('%Y%m%d_%H%M%S')}.zip"
with zipfile.ZipFile(nombre_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(archivo_index, "index.html")

files.download(nombre_zip)
print(f"📦 ZIP descargado: {nombre_zip}")

print("\n" + "="*70)
print("✨ ¡LISTO! NUMERO DE RECARGA CORREGIDO")
print("="*70)
print("✅ CAMBIOS REALIZADOS:")
print("   • ✅ Mapeo corregido para 'Numero de  Recarga' (con espacio)")
print("   • ✅ Popup ahora muestra: Código, N° Recarga, Venta")
print("   • ✅ Ruta optimizada visible y clickeable")
print("   • ✅ Filtros funcionales")
print()
print("📤 PARA DESPLEGAR:")
print("   1. Ve a https://app.netlify.com/drop")
print("   2. Arrastra la carpeta:", carpeta_netlify)
print("="*70)

📂 CARGA DE ARCHIVO EXCEL - CORRECCIÓN NUMERO DE RECARGA

📤 Selecciona tu archivo Excel:


Saving Rutas y frecuencias CM ALL COMO.xlsx to Rutas y frecuencias CM ALL COMO (5).xlsx

✅ 6560 PDV cargados correctamente

✅ ARCHIVO CREADO CON NUMERO DE RECARGA: sitio_mapa_pdv_recarga_ok/index.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📦 ZIP descargado: Mapa_PDV_Recarga_OK_20260502_173313.zip

✨ ¡LISTO! NUMERO DE RECARGA CORREGIDO
✅ CAMBIOS REALIZADOS:
   • ✅ Mapeo corregido para 'Numero de  Recarga' (con espacio)
   • ✅ Popup ahora muestra: Código, N° Recarga, Venta
   • ✅ Ruta optimizada visible y clickeable
   • ✅ Filtros funcionales

📤 PARA DESPLEGAR:
   1. Ve a https://app.netlify.com/drop
   2. Arrastra la carpeta: sitio_mapa_pdv_recarga_ok
